# 03 · Git 工作流实操

> **学习目标**：在一个隔离沙箱里，把 init → commit → branch → merge → rebase → 冲突解决 全跑一遍。
>
> **预备**：本机装了 git；会用基本命令行。
>
> **为什么重要**：你后面所有项目（rag_project / 微调脚本 / Skill 仓）都要 git。出 bug 一半是 git 操作不熟把代码搞丢了。

In [ ]:
import subprocess, shutil, os, stat
from pathlib import Path

def _force_remove(func, p, _exc):
    """rmtree onerror：Windows 上 .git 里的 pack 文件可能是只读，需要先 chmod。"""
    try:
        os.chmod(p, stat.S_IWRITE)
        func(p)
    except Exception:
        pass

SANDBOX = Path('./_git_sandbox').resolve()
if SANDBOX.exists():
    shutil.rmtree(SANDBOX, onerror=_force_remove)
SANDBOX.mkdir()
os.chdir(SANDBOX)
print('cwd =', os.getcwd())

def sh(cmd: str, check: bool = True) -> str:
    """在沙箱里跑 shell 命令并打印输出。Windows 上强制 UTF-8 防止 git 中文输出炸 GBK。"""
    print(f'$ {cmd}')
    res = subprocess.run(cmd, shell=True, capture_output=True,
                          text=True, encoding='utf-8', errors='replace')
    out = ((res.stdout or '') + (res.stderr or '')).rstrip()
    if out:
        print(out)
    if check and res.returncode != 0:
        raise RuntimeError(f'命令失败 (exit {res.returncode})')
    return out

sh('git --version')

## 1. init + 3 次 commit

**关键概念**：
- `git init` 在当前目录建一个 `.git/` —— 这才是仓库的本体
- 工作区（文件本身）→ 暂存区（`git add`）→ 仓库（`git commit`）三层概念，别混

In [ ]:
sh('git init -b main')                       # 直接以 main 为初始分支
sh('git config user.email "you@example.com"')
sh('git config user.name  "learner"')

Path('hello.txt').write_text('Hello\n', encoding='utf-8')
sh('git add hello.txt')
sh('git commit -m "feat: add hello"')

Path('note.md').write_text('# Notes\n\n- start\n', encoding='utf-8')
sh('git add note.md')
sh('git commit -m "docs: add note"')

Path('hello.txt').write_text('Hello, World\n', encoding='utf-8')
sh('git add hello.txt')
sh('git commit -m "feat: greet the world"')

sh('git log --oneline')

## 2. Branch & Merge

**心智模型**：
- 分支只是「指向某个 commit 的可移动指针」
- `merge` 会保留两个分支的历史，常常生成一个 merge commit

In [ ]:
# 从 main 开一个 feature 分支
sh('git switch -c feature/upper')
Path('hello.txt').write_text('HELLO, WORLD\n', encoding='utf-8')
sh('git commit -am "feat: shout it"')

# 回 main，merge feature
sh('git switch main')
sh('git merge feature/upper --no-ff -m "merge: feature/upper"')
sh('git log --oneline --graph --all')

## 3. Rebase — 把分支线性化

**对比 merge**：merge 保留分叉，rebase 把分叉「拍直」成一条线。

**何时用 rebase**：本地分支、还没 push、想让历史干净。
**何时不用**：已经 push 给别人的分支 —— rebase 会改 commit 哈希，别人会爆炸。

下面演示：在 main 上又前进了一步、同时 side 分支也前进了一步，用 rebase 把 side 接到 main 后面。

In [ ]:
# 准备：从当前 main 拉 side 分支，同时 main 也再前进一步
sh('git switch -c side')
Path('side.txt').write_text('side\n', encoding='utf-8')
sh('git add side.txt')
sh('git commit -m "feat(side): add side.txt"')

sh('git switch main')
Path('main.txt').write_text('main\n', encoding='utf-8')
sh('git add main.txt')
sh('git commit -m "feat(main): add main.txt"')

sh('git log --oneline --graph --all')
print('\n--- 此时 main 和 side 各往前一步，是分叉的 ---')

In [ ]:
# 切到 side，rebase 到 main 上 —— 把 side 的 commit 「搬」到 main 最新 commit 之上
sh('git switch side')
sh('git rebase main')
sh('git log --oneline --graph --all')
print('\n--- 现在 side 在 main 之后，历史是一条直线 ---')

## 4. 触发并解决冲突

**冲突的本质**：两个分支改了**同一文件的同一行**，git 不知道你要保留哪个版本。

**解决三步**：
1. `git status` 看哪些文件冲突
2. 打开文件，找到 `<<<<<<<` `=======` `>>>>>>>` 标记，**手动编辑成你想要的最终版本**（含完全删除这些标记）
3. `git add` 标记冲突已解决，然后 `git rebase --continue` 或 `git merge --continue`

In [ ]:
# 回 main，再开 conflict 分支，俩都改 hello.txt 同一行
sh('git switch main')
sh('git switch -c conflict')
Path('hello.txt').write_text('HELLO from conflict branch\n', encoding='utf-8')
sh('git commit -am "chore: edit hello in conflict"')

sh('git switch main')
Path('hello.txt').write_text('HELLO from main branch\n', encoding='utf-8')
sh('git commit -am "chore: edit hello in main"')

# 现在 merge conflict 进 main —— 必冲突
sh('git merge conflict', check=False)
print('--- 看 hello.txt 里的冲突标记 ---')
print(Path('hello.txt').read_text(encoding='utf-8'))

In [ ]:
# 模拟人工解决：写一份「融合」版本，删掉冲突标记
Path('hello.txt').write_text('HELLO from BOTH branches\n', encoding='utf-8')
sh('git add hello.txt')
sh('git commit -m "merge: resolve hello conflict"')
sh('git log --oneline --graph --all')

## 5. 救命级常用命令

**记不住这几条 → 后面你会丢代码**。

In [ ]:
# 看当前状态
sh('git status')

# 看美观分支图（请背下来）
sh('git log --oneline --graph --decorate --all -n 20')

# 临时藏起改动（切分支前救急）
Path('temp.txt').write_text('wip\n', encoding='utf-8')
sh('git add temp.txt')
sh('git stash push -m "wip: temp"')
sh('git stash list')
sh('git stash pop')                          # 恢复

# 看某次 commit 改了什么
sh('git show HEAD --stat')

In [ ]:
# reset 三种模式 —— 区别一定要搞清
# --soft : 只挪 HEAD 指针，暂存区和工作区都保留
# --mixed: 挪 HEAD + 重置暂存区，工作区保留（默认）
# --hard : 全部重置 ⚠️ 工作区改动会丢，做之前必须 stash 或 commit

print('当前 HEAD:')
sh('git log --oneline -n 3')

# 演示 --soft 把最近一次合并 commit 撤回，但代码留在暂存区
sh('git reset --soft HEAD~1')
print('\n--soft 后：')
sh('git log --oneline -n 3')
sh('git status -sb')

# 想回去？用 reflog 找回任何「失踪」的 commit
sh('git reflog -n 5')
print('\n→ reflog 是最后救命稻草。任何「丢」掉的 commit，只要还没 gc，都在 reflog 里找得到。')

## 深入思考

1. **`merge` vs `rebase` 该选哪个？**
   - 团队公共分支：`merge`，保留真实历史
   - 自己本地分支：`rebase`，让历史干净
   - **铁律**：已经 push 过的分支不要 rebase
2. **`git reset --hard` 之后还能救回来吗？**
   - 能。`git reflog` 找到那条丢失 commit 的 hash，`git reset --hard <hash>` 回去。**90 天内** 不会被 gc。
3. **`.gitignore` 加晚了，已经 commit 进去的怎么办？**
   - `git rm --cached <file>` 让 git 不再追踪，再 commit。文件本身保留在工作区。
4. **commit message 写错了？**
   - 最近一次：`git commit --amend -m "new msg"`（**已 push 别这么干**）
   - 老的：`git rebase -i HEAD~N` 改

试试：在沙箱里故意 `git reset --hard HEAD~3` 把最近 3 个 commit 「删掉」，然后用 reflog 找回来。

## 自检 ✅

- [ ] 解释「工作区 / 暂存区 / 仓库」三层结构。
- [ ] 解释 `merge` 与 `rebase` 各自适合什么场景。
- [ ] 现场模拟一次冲突并解决（在你自己电脑上随手做一次）。
- [ ] 当老板说「我刚 `reset --hard` 了，代码没了」，能 30 秒内告诉他用 `reflog`。
- [ ] 解释「为什么已 push 的分支不能 rebase」。

In [ ]:
# 沙箱清理（看完想留着研究就跳过这格）
os.chdir('..')
shutil.rmtree(SANDBOX, onerror=_force_remove)
print('沙箱已清理')

## 下一步

→ [`04_vector_math.ipynb`](04_vector_math.ipynb)